# Análisis descriptivo del dataset IC-UNED-RC-ES

Este notebook describe los dos ficheros que componen el corpus de comprensión lectora del Instituto Cervantes / UNED:

| Fichero | Tipo de tarea |
|---------|--------------|
| `multiple_choice_dataset.json` | Selección múltiple (*multiple-choice*) |
| `matching_dataset.json` | Emparejamiento (*matching*) |

El objetivo es documentar la distribución por nivel y subtipo de ejercicio para justificar decisiones de diseño técnico posteriores (p. ej. pipelines diferenciados, estrategias de prompting, manejo de imágenes).


## 0. Carga de datos

In [4]:
import json
from collections import Counter, defaultdict

MC_PATH = "../../data/test/multiple_choice_dataset.json"
MT_PATH = "../../data/test/matching_dataset.json"

with open(MC_PATH) as f:
    mc_raw = json.load(f)
with open(MT_PATH) as f:
    mt_raw = json.load(f)

print("multiple_choice — acrónimo:", mc_raw["dataset-acronym"])
print("matching        — acrónimo:", mt_raw["dataset-acronym"])
print()
print("Descripción (ES):")
print(" ", mc_raw["dataset-desc-es"])
print(" ", mt_raw["dataset-desc-es"])


multiple_choice — acrónimo: IC-UNED-RC-ES
matching        — acrónimo: IC-UNED-RC-ES

Descripción (ES):
  Corpus de las tareas de comprensión lectora contenidas en los exámenes creados por el Instituto Cervantes para la evaluación de estudiantes de español en diferentes niveles
  Corpus de las tareas de comprensión lectora contenidas en los exámenes creados por el Instituto Cervantes para la evaluación de estudiantes de español en diferentes niveles


## 1. Resumen global

In [5]:
def flatten_exercises(raw):
    rows = []
    for exam in raw["exams"]:
        for ex in exam["exercises"]:
            rows.append({"level": exam["level"], "examId": exam["examId"], **ex})
    return rows

mc_exs = flatten_exercises(mc_raw)
mt_exs = flatten_exercises(mt_raw)

mc_exams_n = len(mc_raw["exams"])
mt_exams_n = len(mt_raw["exams"])

print(f"{'':30} {'multiple-choice':>18} {'matching':>12}")
print("-" * 62)
print(f"{'Exámenes únicos':30} {mc_exams_n:>18} {mt_exams_n:>12}")
print(f"{'Ejercicios':30} {len(mc_exs):>18} {len(mt_exs):>12}")
print(f"{'Preguntas totales':30} {sum(e['num-questions'] for e in mc_exs):>18} {sum(e['num-questions'] for e in mt_exs):>12}")


                                  multiple-choice     matching
--------------------------------------------------------------
Exámenes únicos                               164          126
Ejercicios                                    721          189
Preguntas totales                            1918         1394


## 2. Distribución por nivel

Los niveles siguen el Marco Común Europeo de Referencia (MCER). Los sufijos **E** (*Escolar*) y **A2B1E** indican variantes escolares de los exámenes DELE, orientadas a estudiantes en edad escolar, con textos y registros adaptados.


In [6]:
LEVEL_ORDER = ["A1", "A1E", "A2", "A2B1E", "B1", "B1E", "B2", "C1", "C2"]

def level_table(exs, label):
    lc = Counter(e["level"] for e in exs)
    lq = defaultdict(int)
    for e in exs:
        lq[e["level"]] += e["num-questions"]
    print(f"\n{'--- ' + label + ' ---':}")
    print(f"  {'Nivel':<10} {'Ejercicios':>12} {'Preguntas':>12}  {'Preg/ejercicio':>16}")
    print("  " + "-" * 54)
    for lvl in LEVEL_ORDER:
        if lvl in lc:
            ratio = lq[lvl] / lc[lvl]
            print(f"  {lvl:<10} {lc[lvl]:>12} {lq[lvl]:>12}  {ratio:>15.1f}")
    print(f"  {'TOTAL':<10} {len(exs):>12} {sum(e['num-questions'] for e in exs):>12}")

level_table(mc_exs, "multiple-choice")
level_table(mt_exs, "matching")



--- multiple-choice ---
  Nivel        Ejercicios    Preguntas    Preg/ejercicio
  ------------------------------------------------------
  A1                   39          201              5.2
  A1E                  14           85              6.1
  A2                  245          488              2.0
  A2B1E                 4           24              6.0
  B1                  215          504              2.3
  B1E                  45          100              2.2
  B2                   97          306              3.2
  C1                    8           48              6.0
  C2                   54          162              3.0
  TOTAL               721         1918

--- matching ---
  Nivel        Ejercicios    Preguntas    Preg/ejercicio
  ------------------------------------------------------
  A1                   56          392              7.0
  A1E                  13           91              7.0
  A2                   54          402              7.4
  A2B1E           

## 3. Estructura interna de los ejercicios

### 3a. Multiple-choice: número de opciones por pregunta

La mayoría de las preguntas ofrecen **3 opciones**; un subconjunto presenta 2 ó 4. Esto tiene implicaciones directas para el cálculo de métricas de evaluación (p. ej. *accuracy* ajustada por azar).


In [7]:
option_counts = []
for ex in mc_exs:
    for q in ex["exercise"]["questions"]:
        option_counts.append(len(q["options"]))

dist_opts = Counter(option_counts)
print("Distribución del nº de opciones por pregunta (multiple-choice):")
print(f"  {'Opciones':>8} {'Preguntas':>12}  {'%':>7}")
total_q = sum(dist_opts.values())
for k in sorted(dist_opts):
    print(f"  {k:>8} {dist_opts[k]:>12}  {dist_opts[k]/total_q*100:>6.1f}%")


Distribución del nº de opciones por pregunta (multiple-choice):
  Opciones    Preguntas        %
         2          154     8.4%
         3         1570    85.8%
         4          106     5.8%


### 3b. Multiple-choice: distribución del tamaño del ejercicio

Un ejercicio puede contener de 1 a 10 preguntas. Los ejercicios de **1 pregunta** son los más frecuentes; corresponden a ítems sueltos que acompañan textos cortos (carteles, tickets, avisos) típicos de los niveles A.


In [8]:
dist_nq_mc = Counter(e["num-questions"] for e in mc_exs)
print("Distribución del nº de preguntas por ejercicio (multiple-choice):")
print(f"  {'Preguntas/ej':>13} {'Ejercicios':>12}  {'%':>7}")
total_ex = len(mc_exs)
for k in sorted(dist_nq_mc):
    print(f"  {k:>13} {dist_nq_mc[k]:>12}  {dist_nq_mc[k]/total_ex*100:>6.1f}%")


Distribución del nº de preguntas por ejercicio (multiple-choice):
   Preguntas/ej   Ejercicios        %
              1          387    53.7%
              3          174    24.1%
              5           73    10.1%
              6           53     7.4%
              8            7     1.0%
             10           27     3.7%


### 3c. Matching: tamaño de los conjuntos set1 / set2

En los ejercicios de emparejamiento, **set1** contiene los elementos a emparejar (textos largos o imágenes) y **set2** las preguntas. Cuando set1 > set2 existen **distractores** (opciones de set1 que no son respuesta correcta de ninguna pregunta).


In [9]:
set1_sizes = Counter(len(ex["exercise"]["set1"]) for ex in mt_exs)
nq_mt = Counter(e["num-questions"] for e in mt_exs)

print("Tamaño de set1 (nº de candidatos/textos):")
print(f"  {'set1 size':>10} {'Ejercicios':>12}  {'%':>7}")
for k in sorted(set1_sizes):
    print(f"  {k:>10} {set1_sizes[k]:>12}  {set1_sizes[k]/len(mt_exs)*100:>6.1f}%")

print()
print("Distribución del nº de preguntas por ejercicio (matching):")
print(f"  {'Preguntas/ej':>13} {'Ejercicios':>12}  {'%':>7}")
for k in sorted(nq_mt):
    print(f"  {k:>13} {nq_mt[k]:>12}  {nq_mt[k]/len(mt_exs)*100:>6.1f}%")


Tamaño de set1 (nº de candidatos/textos):
   set1 size   Ejercicios        %
           3           27    14.3%
           4            4     2.1%
           6            6     3.2%
          10          126    66.7%
          11           26    13.8%

Distribución del nº de preguntas por ejercicio (matching):
   Preguntas/ej   Ejercicios        %
              6           27    14.3%
              7          108    57.1%
              8           32    16.9%
             10           22    11.6%


## 4. Presencia de imágenes

Las imágenes son relevantes para el diseño del sistema: los ejercicios que las contienen **no pueden resolverse solo con texto** y requieren un pipeline multimodal o ser excluidos de benchmarks puramente textuales.


In [10]:
# Multiple-choice: imagen en el texto del ejercicio o en alguna pregunta/opción
mc_img_text = sum(1 for ex in mc_exs if ex["exercise"].get("image-path", ""))
mc_img_q    = sum(1 for ex in mc_exs
                  if any(q.get("image-path","") for q in ex["exercise"]["questions"]))
mc_img_opt  = sum(1 for ex in mc_exs
                  if any(opt.get("image-path","")
                         for q in ex["exercise"]["questions"]
                         for opt in q["options"]))

print("Imágenes en multiple-choice:")
print(f"  En el texto del ejercicio : {mc_img_text:>4} / {len(mc_exs)}")
print(f"  En alguna pregunta        : {mc_img_q:>4} / {len(mc_exs)}")
print(f"  En alguna opción          : {mc_img_opt:>4} / {len(mc_exs)}")

# Matching: set1 o set2
mt_img_s1 = sum(1 for ex in mt_exs if any(x.get("image-path","") for x in ex["exercise"]["set1"]))
mt_img_s2 = sum(1 for ex in mt_exs if any(x.get("image-path","") for x in ex["exercise"]["set2"]))
mt_img_any = sum(1 for ex in mt_exs
                 if any(x.get("image-path","") for x in ex["exercise"]["set1"])
                    or any(x.get("image-path","") for x in ex["exercise"]["set2"]))

print()
print("Imágenes en matching:")
print(f"  En set1 (candidatos)      : {mt_img_s1:>4} / {len(mt_exs)}")
print(f"  En set2 (preguntas)       : {mt_img_s2:>4} / {len(mt_exs)}")
print(f"  En cualquiera de los dos  : {mt_img_any:>4} / {len(mt_exs)}")


Imágenes en multiple-choice:
  En el texto del ejercicio :    0 / 721
  En alguna pregunta        :    0 / 721
  En alguna opción          :   46 / 721

Imágenes en matching:
  En set1 (candidatos)      :   27 / 189
  En set2 (preguntas)       :   44 / 189
  En cualquiera de los dos  :   71 / 189


## 5. Tabla resumen para la descripción oficial del dataset

Cifras consolidadas listas para incluir en la sección de descripción del dataset (paper, README o tarjeta de modelo).


In [11]:
LEVEL_ORDER = ["A1", "A1E", "A2", "A2B1E", "B1", "B1E", "B2", "C1", "C2"]

mc_lc = Counter(e["level"] for e in mc_exs)
mc_lq = defaultdict(int)
for e in mc_exs: mc_lq[e["level"]] += e["num-questions"]

mt_lc = Counter(e["level"] for e in mt_exs)
mt_lq = defaultdict(int)
for e in mt_exs: mt_lq[e["level"]] += e["num-questions"]

all_levels = sorted(set(mc_lc) | set(mt_lc), key=lambda x: LEVEL_ORDER.index(x) if x in LEVEL_ORDER else 99)

print(f"{'Nivel':<10} | {'MC ej':>7} {'MC preg':>9} | {'MT ej':>7} {'MT preg':>9} | {'Total ej':>9} {'Total preg':>11}")
print("-" * 72)
tot_mc_e = tot_mc_q = tot_mt_e = tot_mt_q = 0
for lvl in all_levels:
    me, mq = mc_lc.get(lvl, 0), mc_lq.get(lvl, 0)
    te, tq = mt_lc.get(lvl, 0), mt_lq.get(lvl, 0)
    tot_mc_e += me; tot_mc_q += mq
    tot_mt_e += te; tot_mt_q += tq
    print(f"{lvl:<10} | {me:>7} {mq:>9} | {te:>7} {tq:>9} | {me+te:>9} {mq+tq:>11}")
print("-" * 72)
print(f"{'TOTAL':<10} | {tot_mc_e:>7} {tot_mc_q:>9} | {tot_mt_e:>7} {tot_mt_q:>9} | {tot_mc_e+tot_mt_e:>9} {tot_mc_q+tot_mt_q:>11}")


Nivel      |   MC ej   MC preg |   MT ej   MT preg |  Total ej  Total preg
------------------------------------------------------------------------
A1         |      39       201 |      56       392 |        95         593
A1E        |      14        85 |      13        91 |        27         176
A2         |     245       488 |      54       402 |       299         890
A2B1E      |       4        24 |      12        76 |        16         100
B1         |     215       504 |      26       165 |       241         669
B1E        |      45       100 |       0         0 |        45         100
B2         |      97       306 |       4        40 |       101         346
C1         |       8        48 |       4        32 |        12          80
C2         |      54       162 |      20       196 |        74         358
------------------------------------------------------------------------
TOTAL      |     721      1918 |     189      1394 |       910        3312


## 6. Observaciones de diseño

A partir del análisis anterior, se destacan las siguientes consideraciones para el diseño del sistema:

1. **Dos pipelines distintos.** Multiple-choice y matching tienen estructuras de entrada/salida diferentes (preguntas con opciones vs. emparejamiento de conjuntos) y deben procesarse por separado.

2. **Heterogeneidad en el tamaño de ejercicio (MC).** El 53 % de los ejercicios de selección múltiple contiene una única pregunta; el resto puede llegar a 10. Un enfoque que agrupe todas las preguntas de un ejercicio en un solo prompt es más robusto que procesar pregunta a pregunta.

3. **Distractores en matching.** Cuando `set1_size > num-questions` existen opciones que no son respuesta correcta de ninguna pregunta. El modelo debe discriminar activamente, no solo ordenar.

4. **Niveles escolares (A1E, A2B1E, B1E).** Representan un subconjunto significativo del corpus. Su comportamiento puede diferir del DELE estándar; conviene evaluar por separado.

5. **Imágenes: subconjunto minoritario pero no trivial.** El 37 % de los ejercicios de matching contienen imágenes; en multiple-choice no se encontraron imágenes asociadas en este análisis. Los ejercicios con imágenes en set2 (preguntas ilustradas) son los más exigentes para un sistema basado solo en texto.

6. **Distribución de niveles desequilibrada.** En MC predominan A2 y B1; en matching, A1, A2 y C2. Al reportar métricas globales conviene acompañarlas de resultados por nivel.
